In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Move Detection Tables from dev.detection_old to dev.detection
# MAGIC
# MAGIC This notebook moves the following tables:
# MAGIC - viewing_content_golden
# MAGIC - viewing_content_golden_smoothed
# MAGIC - content_golden_wdr_merge

from pyspark.sql import functions as F
from delta.tables import DeltaTable
import json

In [0]:
# Configuration
source_catalog = "dev"
source_database = "detection_old"
target_catalog = "dev"
target_database = "detection"

tables_to_migrate = [
    "viewing_content_golden",
    "viewing_content_golden_smoothed",
    "content_golden_wdr_merge",
]

In [0]:
def get_table_properties(catalog, database, table):
    """Get table properties and metadata"""
    try:
        properties = spark.sql(f"DESCRIBE TABLE EXTENDED {catalog}.{database}.{table}")
        return properties.collect()
    except Exception as e:
        print(f"Error getting properties for {catalog}.{database}.{table}: {e}")
        return None

In [0]:
def get_table_schema(catalog, database, table):
    """Get table schema"""
    try:
        return spark.table(f"{catalog}.{database}.{table}").schema
    except Exception as e:
        print(f"Error getting schema for {catalog}.{database}.{table}: {e}")
        return None

In [0]:
def copy_table_structure_and_data(
    source_catalog, source_db, target_catalog, target_db, table_name
):
    """Copy table structure and data"""
    source_table = f"{source_catalog}.{source_db}.{table_name}"
    target_table = f"{target_catalog}.{target_db}.{table_name}"

    print(f"Migrating {source_table} -&gt; {target_table}")

    try:
        # Check if target table exists and drop it
        try:
            spark.sql(f"DROP TABLE IF EXISTS {target_table}")
            print(f"Dropped existing table {target_table}")
        except:
            pass

        # Get source table data
        source_df = spark.table(source_table)
        row_count = source_df.count()
        print(f"Source table has {row_count:,} rows")

        # Create target table by writing the data
        (
            source_df.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        # Verify the copy
        target_row_count = spark.table(target_table).count()
        print(f"Target table created with {target_row_count:,} rows")

        if row_count == target_row_count:
            print(f"Successfully migrated {table_name}")
            return True
        else:
            print(
                f"Row count mismatch for {table_name}: source={row_count}, target={target_row_count}"
            )
            return False

    except Exception as e:
        print(f"Error migrating {table_name}: {e}")
        return False

In [0]:
# DBTITLE 1,Migrate Tables
migration_results = {}

for table_name in tables_to_migrate:
    print(f"\n{'='*60}")
    print(f"MIGRATING TABLE: {table_name}")
    print(f"{'='*60}")
    
    success = copy_table_structure_and_data(
        source_catalog, source_database, 
        target_catalog, target_database, 
        table_name
    )
    
    migration_results[table_name] = success

In [0]:
# DBTITLE 1,Migration Summary
print("\n" + "="*60)
print("MIGRATION SUMMARY")
print("="*60)

all_successful = True
for table_name, success in migration_results.items():
    status = "SUCCESS" if success else "FAILED"
    print(f"{table_name}: {status}")
    if not success:
        all_successful = False

if all_successful:
    print(f"\nAll tables successfully migrated to {target_catalog}.{target_database}")
else:
    print(f"\nSome tables failed to migrate. Please check the logs above.")

In [0]:
# DBTITLE 1,Verify Table Access
print("\n" + "="*60)
print("VERIFYING TABLE ACCESS")
print("="*60)

for table_name in tables_to_migrate:
    try:
        target_table = f"{target_catalog}.{target_database}.{table_name}"
        df = spark.table(target_table)
        count = df.count()
        print(f"{target_table}: {count:,} rows - ACCESSIBLE")
    except Exception as e:
        print(f"{target_table}: ERROR - {e}")

In [0]:
# Apply Tags to Migrated Tables
def apply_table_tags(catalog, database, table_name, tags):
    """Apply tags to a table"""
    try:
        for tag_key, tag_value in tags.items():
            spark.sql(f"ALTER TABLE {catalog}.{database}.{table_name} SET TAGS ('{tag_key}' = '{tag_value}')")
        print(f"Applied tags to {catalog}.{database}.{table_name}")
        return True
    except Exception as e:
        print(f"Error applying tags to {table_name}: {e}")
        return False

In [0]:
# Define tags for each table
table_tags = {
    "viewing_content_golden": {
        "environment": "dev",
        "data_domain": "content_viewing", 
        "data_classification": "internal",
        "pipeline_stage": "golden",
        "data_product": "viewing_analytics",
        "owner_team": "data_engineering",
        "retention_period": "2_years",
        "update_frequency": "streaming",
        "pii_flag": "false",
        "business_unit": "content",
        "source_system": "firehose"
    },
    
    "viewing_content_golden_smoothed": {
        "environment": "dev",
        "data_domain": "content_viewing",
        "data_classification": "internal", 
        "pipeline_stage": "golden_processed",
        "data_product": "viewing_analytics",
        "owner_team": "data_engineering",
        "retention_period": "2_years",
        "update_frequency": "hourly_batch",
        "pii_flag": "false",
        "business_unit": "content",
        "source_system": "smoothing_pipeline",
        "processing_type": "smoothed_aggregation"
    },
    
    "content_golden_wdr_merge": {
        "environment": "dev",
        "data_domain": "content_viewing",
        "data_classification": "internal",
        "pipeline_stage": "control_table", 
        "data_product": "viewing_analytics",
        "owner_team": "data_engineering",
        "retention_period": "1_year",
        "update_frequency": "streaming",
        "pii_flag": "false",
        "business_unit": "content", 
        "table_type": "watermark_tracking",
        "processing_type": "pipeline_control"
    }
}

In [0]:
# Apply tags to all migrated tables
print("APPLYING TAGS TO MIGRATED TABLES")
print("="*60)

for table_name in tables_to_migrate:
    if table_name in table_tags:
        tags = table_tags[table_name]
        apply_table_tags(target_catalog, target_database, table_name, tags)
    else:
        print(f"No tags defined for {table_name}")

In [0]:
# DBTITLE 1,Verify Applied Tags
print("\n" + "="*60)
print("VERIFYING APPLIED TAGS")
print("="*60)

for table_name in tables_to_migrate:
    try:
        target_table = f"{target_catalog}.{target_database}.{table_name}"
        
        # Get table information including tags
        table_info = spark.sql(f"DESCRIBE TABLE EXTENDED {target_table}").collect()
        
        print(f"\nTags for {target_table}:")
        
        # Look for tags in the extended information
        tags_found = False
        for row in table_info:
            if row['col_name'] and 'tag' in row['col_name'].lower():
                print(f"  {row['col_name']}: {row['data_type']}")
                tags_found = True

        if not tags_found:
            print("  No tags found in DESCRIBE output")

    except Exception as e:
        print(f"Error checking tags for {table_name}: {e}")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev.detection_old.viewing_content_golden (
  tvid STRING,
  fk_tvid INT,
  zipcode STRING,
  dma STRING,
  tms_episode_id STRING,
  tivo_episode_id STRING,
  tms_title STRING,
  tivo_title STRING,
  tms_airdate TIMESTAMP,
  tivo_airdate TIMESTAMP,
  tms_channel_callsign STRING,
  tivo_channel_callsign STRING,
  mt_start INT,
  session_start TIMESTAMP,
  session_end TIMESTAMP,
  tms_channel_affiliate STRING,
  tivo_channel_affiliate STRING,
  is_live STRING,
  ip_address STRING,
  input_category STRING,
  input_device STRING,
  app_service STRING,
  tuner_tms_episode_id STRING,
  tuner_tivo_episode_id STRING,
  tuner_tms_title STRING,
  tuner_tivo_title STRING,
  tuner_tms_airdate TIMESTAMP,
  tuner_tivo_airdate TIMESTAMP,
  tuner_tms_channel_callsign STRING,
  tuner_tivo_channel_callsign STRING,
  tuner_mt_start INT,
  tuner_tms_channel_affiliate STRING,
  tuner_tivo_channel_affiliate STRING,
  tuner_is_live STRING,
  tuner_input_category STRING,
  tuner_input_device STRING,
  tuner_app_service STRING,
  tuner_channel_number STRING,
  enableaudioacr STRING,
  dma_code INT,
  vizio_epg_channel_id BIGINT,
  vizio_epg_program_id BIGINT,
  tms_show_genre STRING,
  tivo_show_genre STRING,
  tms_epi_title STRING,
  tivo_epi_title STRING,
  series_id STRING,
  show_duration INT,
  vizio_epg_not_null BOOLEAN,
  nielsen_exclusive BOOLEAN,
  content_only_condition BOOLEAN,
  tuner_content_only_condition BOOLEAN,
  vod_station BOOLEAN,
  acrb_clients STRING,
  appb_clients STRING,
  client_id_not_null STRING,
  session_start_hour STRING)
  USING delta
PARTITIONED BY (session_start_hour)
LOCATION 's3://inscape-databricks-dev/databases/dev_detection.db/detection/data/viewing_content_golden';

In [0]:
%sql
SELECT COUNT(*) FROM dev.detection